In [1]:
# ============================================================
# SUPPLY CHAIN INTELLIGENCE SYSTEM
# Notebook 8: Load All Data into PostgreSQL
# ============================================================

import sys
!{sys.executable} -m pip install psycopg2-binary sqlalchemy --quiet

import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries ready!")

✅ Libraries ready!


In [7]:
# ============================================================
# Connect to PostgreSQL
# Special characters in password handled correctly
# ============================================================

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

DB_USER     = 'postgres'
DB_PASSWORD = quote_plus('NewStrongPassword@123')  # Encodes @ safely
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'supply_chain_db'

try:
    engine = create_engine(
        f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
        connect_args={'connect_timeout': 10}
    )
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("✅ Connected to PostgreSQL successfully!")
        print(f"   {result.fetchone()[0][:60]}")

except Exception as e:
    print(f"❌ Connection failed!")
    print(f"   Error: {str(e)[:200]}")

✅ Connected to PostgreSQL successfully!
   PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35


In [8]:
# ============================================================
# Load all cleaned CSV files into PostgreSQL
# ============================================================

processed_path = r'D:\Projects\End-to-end projects\8. Supply Chain Intelligence\Data\Processed'

# Define all files and their target tables
tables_to_load = [
    ('products_clean.csv',            'products'),
    ('suppliers_clean.csv',           'suppliers'),
    ('purchase_orders_clean.csv',     'purchase_orders'),
    ('inventory_snapshots_clean.csv', 'inventory_snapshots'),
    ('sales_orders_clean.csv',        'sales_orders'),
    ('supplier_scorecard.csv',        'supplier_scorecard'),
    ('disruption_scenarios.csv',      'disruption_scenarios'),
]

print("🔄 Loading data into PostgreSQL...\n")
print(f"{'Table':<30} {'Rows':>8} {'Status'}")
print("-" * 50)

for filename, table_name in tables_to_load:
    try:
        # Load CSV
        df = pd.read_csv(f'{processed_path}\\{filename}')

        # Clean column names — lowercase, no spaces
        df.columns = df.columns.str.lower().str.replace(' ', '_')

        # Load to PostgreSQL — replace if exists
        df.to_sql(
            table_name,
            engine,
            if_exists = 'replace',
            index     = False,
            chunksize = 500,
            method    = 'multi'
        )

        print(f"{table_name:<30} {len(df):>8,}  ✅ Loaded")

    except Exception as e:
        print(f"{table_name:<30} {'ERROR':>8}  ❌ {str(e)[:60]}")

print(f"\n✅ All tables loaded into supply_chain_db!")

🔄 Loading data into PostgreSQL...

Table                              Rows Status
--------------------------------------------------
products                          ERROR  ❌ (psycopg2.errors.DependentObjectsStillExist) cannot drop tab
suppliers                         ERROR  ❌ (psycopg2.errors.DependentObjectsStillExist) cannot drop tab
purchase_orders                   1,960  ✅ Loaded
inventory_snapshots              15,750  ✅ Loaded
sales_orders                      2,600  ✅ Loaded
supplier_scorecard                   15  ✅ Loaded
disruption_scenarios                 40  ✅ Loaded

✅ All tables loaded into supply_chain_db!


In [9]:
# ============================================================
# Fix: Load products and suppliers with dependency handling
# ============================================================

from sqlalchemy import text

with engine.connect() as conn:
    # Disable foreign key checks temporarily
    conn.execute(text("SET session_replication_role = 'replica';"))
    conn.commit()

    for filename, table_name in [('products_clean.csv', 'products'),
                                  ('suppliers_clean.csv', 'suppliers')]:
        try:
            df = pd.read_csv(f'{processed_path}\\{filename}')
            df.columns = df.columns.str.lower().str.replace(' ', '_')

            # Truncate first then insert
            conn.execute(text(f"TRUNCATE TABLE {table_name} CASCADE;"))
            conn.commit()

            df.to_sql(
                table_name,
                engine,
                if_exists = 'append',
                index     = False,
                chunksize = 500,
                method    = 'multi'
            )
            print(f"✅ {table_name:<20} {len(df):>6,} rows loaded")

        except Exception as e:
            print(f"❌ {table_name:<20} Error: {str(e)[:80]}")

    # Re-enable foreign key checks
    conn.execute(text("SET session_replication_role = 'origin';"))
    conn.commit()

print(f"\n✅ Products and suppliers loaded successfully!")

✅ products                 50 rows loaded
✅ suppliers                15 rows loaded

✅ Products and suppliers loaded successfully!
